# Sprint 2: Wikipedia Vector Database Exploration

Explore the ingested Wikipedia data and vector database functionality.

In [ ]:
import sys
import os
import json
import requests
import pandas as pd
from datetime import datetime

# Add src directory to path
sys.path.insert(0, '../src')

print(f"Sprint 2 Data Exploration: {datetime.now().strftime('%H:%M:%S')}")
print(f"Python: {sys.version.split()[0]}")

## 1. Vector Database API Status

In [ ]:
# Check API server status
API_URL = "http://localhost:8001"

try:
    # Get system info
    info_response = requests.get(f"{API_URL}/vectordb/info")
    if info_response.status_code == 200:
        info = info_response.json()
        print("📊 Vector Database System Info:")
        print(f"  Description: {info['description']}")
        print(f"  Embedding Model: {info['embedding_model']}")
        print(f"  Available Endpoints: {list(info['endpoints'].keys())}")
    
    # Get current status
    status_response = requests.get(f"{API_URL}/vectordb/status")
    if status_response.status_code == 200:
        status = status_response.json()
        print("\n✅ Current Database Status:")
        for key, value in status.items():
            if key != 'timestamp':
                print(f"  {key}: {value}")
    
except requests.exceptions.RequestException as e:
    print(f"❌ API connection failed: {e}")
    print("Make sure the API server is running on port 8001")

## 2. Explore Ingested Documents

In [ ]:
# Load and examine the document store
try:
    with open('../src/storage/vector_index/docstore.json', 'r') as f:
        docstore_data = json.load(f)
    
    doc_data = docstore_data['docstore/data']
    ref_doc_info = docstore_data['docstore/ref_doc_info']
    
    print(f"📚 Total Documents Loaded: {len(ref_doc_info)}")
    print(f"📄 Total Text Nodes: {len(doc_data)}")
    
    # Show sample documents
    print("\n📖 Sample Wikipedia Articles:")
    print("=" * 50)
    
    for i, (ref_id, ref_info) in enumerate(list(ref_doc_info.items())[:5]):
        node_id = ref_info['node_ids'][0]
        if node_id in doc_data:
            node = doc_data[node_id]['__data__']
            title = node['metadata'].get('title', 'Unknown')
            text = node.get('text', '').strip()
            doc_id = node['metadata'].get('doc_id', 'Unknown')
            
            print(f"\nArticle {i+1} (ID: {doc_id})")
            print(f"Title: {title}")
            print(f"Length: {len(text)} characters")
            print(f"Content: {text[:200]}..." if len(text) > 200 else f"Content: {text}")
            
except FileNotFoundError:
    print("❌ Document store not found. Run ingestion first.")
except Exception as e:
    print(f"❌ Error reading document store: {e}")

## 3. Vector Database Analysis

In [ ]:
# Examine vector embeddings
try:
    with open('../src/storage/vector_index/default__vector_store.json', 'r') as f:
        vector_data = json.load(f)
    
    embeddings = vector_data.get('embedding_dict', {})
    
    print(f"🔢 Vector Database Statistics:")
    print(f"  Total Embeddings: {len(embeddings)}")
    
    if embeddings:
        # Get first embedding to check dimensions
        first_embedding = list(embeddings.values())[0]
        print(f"  Embedding Dimensions: {len(first_embedding)}")
        print(f"  Sample Values: {first_embedding[:5]}...")
        
        # Calculate some statistics
        import numpy as np
        all_embeddings = np.array(list(embeddings.values()))
        
        print(f"\n📊 Embedding Statistics:")
        print(f"  Mean: {np.mean(all_embeddings):.4f}")
        print(f"  Std Dev: {np.std(all_embeddings):.4f}")
        print(f"  Min: {np.min(all_embeddings):.4f}")
        print(f"  Max: {np.max(all_embeddings):.4f}")
    
except FileNotFoundError:
    print("❌ Vector store not found. Run ingestion first.")
except Exception as e:
    print(f"❌ Error reading vector store: {e}")

## 4. Test Vector Search Functionality

In [ ]:
# Test the ingestion pipeline with correct path
try:
    from ingestion import IngestionPipeline
    
    # Use the correct storage path where data actually exists
    pipeline = IngestionPipeline(storage_dir='../src/storage')
    
    # Try to load existing index
    existing_index = pipeline.load_existing_index()
    if existing_index:
        print("✅ Successfully loaded existing vector index")
        pipeline.index = existing_index
    
    stats = pipeline.get_index_stats()
    
    print("🔧 Pipeline Statistics:")
    for key, value in stats.items():
        print(f"  {key}: {value}")
    
    # Test query if index is available
    if pipeline.index:
        print("\n🔍 Testing Vector Search:")
        test_queries = [
            "What is Uruguay?",
            "Tell me about the economy", 
            "What about the capital city?"
        ]
        
        for query in test_queries:
            print(f"\nQuery: '{query}'")
            response = pipeline.query_index(query, top_k=2)
            if response:
                print(f"Response: {response[:200]}...")
            else:
                print("No response received")
    else:
        print("❌ No vector index available for querying")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
except Exception as e:
    print(f"❌ Pipeline error: {e}")

## 5. Storage Analysis

In [ ]:
import os

# Analyze storage usage
storage_path = '../src/storage/vector_index/'

if os.path.exists(storage_path):
    print("💾 Storage Analysis:")
    
    total_size = 0
    files = os.listdir(storage_path)
    
    for file in files:
        file_path = os.path.join(storage_path, file)
        if os.path.isfile(file_path):
            size = os.path.getsize(file_path)
            total_size += size
            print(f"  {file}: {size / 1024:.1f} KB")
    
    print(f"\n📊 Total Storage: {total_size / 1024:.1f} KB")
    print(f"📈 Storage per Document: {(total_size / len(ref_doc_info)) / 1024:.1f} KB" if 'ref_doc_info' in locals() else "")
else:
    print("❌ Storage directory not found")

## 6. Sprint 2 Summary

In [ ]:
print("🎯 SPRINT 2 DATA EXPLORATION COMPLETE")
print("=" * 50)

achievements = [
    "✅ Wikipedia articles successfully ingested",
    "✅ 3072-dimensional embeddings generated", 
    "✅ Vector database operational and queryable",
    "✅ REST API endpoints functional",
    "✅ Document content and metadata preserved",
    "✅ Semantic search capabilities working"
]

for achievement in achievements:
    print(achievement)

print(f"\n📅 Exploration completed: {datetime.now().isoformat()}")
print("🚀 Ready for Sprint 3: RAG Query Implementation")